# Dan Koe Style — BiRefNet GPU (Google Colab)
**Requiere:** Runtime → Change runtime type → **T4 GPU**

Pipeline:
1. Instala dependencias
2. Sube tu video
3. Procesamiento BiRefNet GPU + eliminación apoyacabezas
4. Descarga el resultado

In [ ]:
# CELDA 1 — Verificar GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'Sin GPU — cambia el runtime a T4 GPU')

In [ ]:
# CELDA 2 — Instalar dependencias
!pip install -q opencv-python-headless numpy pillow tqdm onnxruntime-gpu
!apt-get install -q ffmpeg

import os
os.makedirs('/root/.u2net', exist_ok=True)
if not os.path.exists('/root/.u2net/birefnet-general.onnx'):
    print('Descargando BiRefNet (~973MB)...')
    !wget -q --show-progress -O /root/.u2net/birefnet-general.onnx \
        https://github.com/danielgatis/rembg/releases/download/v0.0.0/BiRefNet-general-epoch_244.onnx
    print('Descarga completa')
else:
    print('Modelo ya descargado')

In [ ]:
# CELDA 3 — Subir video
from google.colab import files
print('Sube reel_test.mp4 (5s) o reel_1080p.mp4 (reel completo)')
uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
print(f'Video subido: {input_filename}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELDA 4 — REELS 9:16  ✅ CONFIGURACIÓN VALIDADA               ║
# ║  BiRefNet GPU · Fondo negro · Borde limpio · Rim light cálido  ║
# ║  SOLO CAMBIAR: INPUT_VIDEO                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

try:
    del session
except NameError:
    pass
import gc; gc.collect()

import cv2
import numpy as np
import onnxruntime as ort
import subprocess, time
from pathlib import Path
from tqdm import tqdm

# ── CAMBIAR ESTO POR CADA REEL ──────────────────────────────────────
INPUT_VIDEO   = '/kaggle/input/datasets/luiscabrejo/reel-1/reel-1_test.mp4'
OUTPUT_VIDEO  = 'video_dankoe.mp4'
# ────────────────────────────────────────────────────────────────────

MODEL_PATH    = '/root/.u2net/birefnet-general.onnx'
OUTPUT_WIDTH  = 1080
OUTPUT_HEIGHT = 1920

BG_CENTER_BRIGHTNESS  = 45
BG_EDGE_BRIGHTNESS    = 0
GRADIENT_FALLOFF      = 2.5
GRADIENT_CENTER_Y     = 0.40
ALPHA_TEMPORAL_WEIGHT = 0.35
SHADOW_BLUE_LIFT      = 8
HIGHLIGHT_WARMTH      = 6
CONTRAST_STRENGTH     = 0.40
MASK_CLOSE_PX         = 6
MASK_ERODE_PX         = 0
MASK_FEATHER_PX       = 2
FADE_START_Y          = 0.78
FB_R1, FB_R2          = 90, 3

SIGMOID_TEMP          = 3.5
HALO_BAND_PX          = 10
HALO_LUM_THRESHOLD    = 0.94
HALO_DESAT_STRENGTH   = 0.50
DESPILL_STRENGTH      = 0.50

RIM_WIDTH             = 8
RIM_INTENSITY         = 0.85
RIM_COLOR_BGR         = (80, 55, 35)

def load_model():
    opts = ort.SessionOptions()
    opts.enable_mem_pattern = False
    opts.enable_cpu_mem_arena = False
    sess = ort.InferenceSession(MODEL_PATH, sess_options=opts,
                                providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
    print(f'Provider: {sess.get_providers()[0]}')
    return sess

def preprocess(img_rgb):
    img = cv2.resize(img_rgb, (1024, 1024), interpolation=cv2.INTER_LINEAR)
    img = (img.astype(np.float32)/255.0 - [0.485,0.456,0.406]) / [0.229,0.224,0.225]
    return img.transpose(2,0,1)[None].astype(np.float32)

def predict_mask(session, img_rgb, orig_w, orig_h):
    out = session.run(None, {session.get_inputs()[0].name: preprocess(img_rgb)})[0]
    pred = 1.0 / (1.0 + np.exp(-out[0,0]))
    mi, ma = pred.min(), pred.max()
    if ma > mi: pred = (pred - mi) / (ma - mi)
    return cv2.resize(pred, (orig_w, orig_h), interpolation=cv2.INTER_LANCZOS4).astype(np.float32)

def sharpen_alpha(alpha, temperature=3.5):
    return (1.0 / (1.0 + np.exp(-temperature * (alpha - 0.5)))).astype(np.float32)

def refine_alpha(alpha, close_px, erode_px, blur_px):
    a = (alpha * 255).astype(np.uint8)
    if close_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_px*2+1, close_px*2+1))
        a = cv2.morphologyEx(a, cv2.MORPH_CLOSE, k)
    if erode_px > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_px*2+1, erode_px*2+1))
        a = cv2.erode(a, k, iterations=1)
    alpha = a.astype(np.float32) / 255.0
    if blur_px > 0:
        alpha = cv2.GaussianBlur(alpha, (blur_px*2+1, blur_px*2+1), 0)
    return alpha

def get_edge_band(alpha, band_px):
    a = (alpha * 255).astype(np.uint8)
    _, binary = cv2.threshold(a, 10, 255, cv2.THRESH_BINARY)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (band_px*2+1, band_px*2+1))
    return cv2.subtract(cv2.dilate(binary, k), cv2.erode(binary, k)).astype(bool)

def white_despill(frame_f32, alpha):
    a3 = alpha[:,:,None]
    transition = (alpha > 0.05) & (alpha < 0.95)
    fg = np.clip((frame_f32 - (1.0 - a3)) / np.maximum(a3, 1e-6), 0, 1)
    return np.where(transition[:,:,None], frame_f32*(1-DESPILL_STRENGTH) + fg*DESPILL_STRENGTH, frame_f32).astype(np.float32)

def remove_halo_dynamic(frame_f32, alpha):
    edge = get_edge_band(alpha, HALO_BAND_PX)
    lum  = 0.299*frame_f32[...,0] + 0.587*frame_f32[...,1] + 0.114*frame_f32[...,2]
    halo = edge & (alpha > 0.05) & (lum > HALO_LUM_THRESHOLD)
    if not np.any(halo):
        return frame_f32, alpha
    frame = frame_f32.copy()
    excess = np.clip(lum - HALO_LUM_THRESHOLD, 0, 1)
    correction = np.clip(1.0 - excess * HALO_DESAT_STRENGTH * 2.0, 0.3, 1.0)
    for c in range(3):
        ch = frame[...,c]; ch[halo] = (ch * correction)[halo]; frame[...,c] = ch
    return frame, alpha

def fb_fusion(image, F, B, alpha, r):
    ba = cv2.blur(alpha[:,:,0], (r,r))[:,:,None]
    bF = cv2.blur(F*alpha, (r,r)) / (ba + 1e-5)
    bB = cv2.blur(B*(1-alpha), (r,r)) / ((1-ba) + 1e-5)
    return np.clip(bF + alpha*(image - alpha*bF - (1-alpha)*bB), 0, 1), bB

def decontaminate(frame_f32, alpha):
    a3 = alpha[:,:,None]
    F, bB = fb_fusion(frame_f32, frame_f32, np.ones_like(frame_f32), a3, FB_R1)
    F, _  = fb_fusion(frame_f32, F, bB, a3, FB_R2)
    return F

def create_gradient(w, h):
    y,x  = np.ogrid[:h,:w]
    cx, cy = w//2, int(h*GRADIENT_CENTER_Y)
    md   = np.sqrt(max(cx,w-cx)**2 + max(cy,h-cy)**2)
    mask = np.clip(1.0 - np.sqrt((x-cx)**2+(y-cy)**2)/md, 0, 1)**GRADIENT_FALLOFF
    br   = BG_EDGE_BRIGHTNESS + (BG_CENTER_BRIGHTNESS-BG_EDGE_BRIGHTNESS)*mask
    return np.clip(np.stack([br*1.05, br*0.95, br*0.90], axis=-1), 0, 255).astype(np.uint8)

def grade(bgr):
    img = bgr.astype(np.float32)
    lum = (0.114*img[...,0]+0.587*img[...,1]+0.299*img[...,2])/255.0
    sh  = np.clip(1-lum/0.35, 0, 1)[...,None]
    hi  = np.clip((lum-0.30)/0.70, 0, 1)[...,None]
    img[...,0] += sh[...,0]*SHADOW_BLUE_LIFT
    img[...,2] += hi[...,0]*HIGHLIGHT_WARMTH - sh[...,0]*(SHADOW_BLUE_LIFT*0.5)
    n = img/255.0
    s = np.where(n<0.5, 2*n**2, 1-2*(1-n)**2)
    return np.clip((n*(1-CONTRAST_STRENGTH)+s*CONTRAST_STRENGTH)*255, 0, 255).astype(np.uint8)

def add_rim_light(canvas, alpha, oy, ox, sh2, sw):
    a_u8 = (alpha*255).astype(np.uint8)
    _, binary = cv2.threshold(a_u8, 10, 255, cv2.THRESH_BINARY)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (RIM_WIDTH*2+1, RIM_WIDTH*2+1))
    rim  = cv2.GaussianBlur(cv2.subtract(cv2.dilate(binary,k), binary).astype(np.float32),
                            (RIM_WIDTH*2+1, RIM_WIDTH*2+1), RIM_WIDTH*0.5)
    region = canvas[oy:oy+sh2, ox:ox+sw].astype(np.float32)
    region += (rim/255.0)[:,:,None] * np.array(RIM_COLOR_BGR, np.float32) * RIM_INTENSITY
    canvas[oy:oy+sh2, ox:ox+sw] = np.clip(region, 0, 255).astype(np.uint8)
    return canvas

def bottom_fade(canvas):
    h  = canvas.shape[0]
    sp = int(h * FADE_START_Y)
    g  = np.linspace(1.0, 0.0, h-sp, dtype=np.float32)
    for i, gv in enumerate(g):
        canvas[sp+i] = (canvas[sp+i].astype(np.float32)*gv).astype(np.uint8)
    return canvas

# ── MAIN ────────────────────────────────────────────────────────────
t0 = time.time()
cap = cv2.VideoCapture(INPUT_VIDEO)
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f'{w}x{h} @ {fps:.1f}fps — {n} frames')

tw, th = OUTPUT_WIDTH, OUTPUT_HEIGHT
sc = min(tw/w, th/h)
sw, sh2 = int(w*sc), int(h*sc)
ox, oy  = (tw-sw)//2, (th-sh2)//2

session    = load_model()
bg         = create_gradient(tw, th)
temp       = 'video_dankoe.temp.mp4'
writer     = cv2.VideoWriter(temp, cv2.VideoWriter_fourcc(*'mp4v'), fps, (tw, th))
prev_alpha = None

print('Procesando frames...')
for _ in tqdm(range(n), unit='fr'):
    ret, frame = cap.read()
    if not ret: break
    fr  = cv2.resize(frame, (sw, sh2), interpolation=cv2.INTER_LINEAR)
    rgb = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
    alpha = predict_mask(session, rgb, sw, sh2)
    if prev_alpha is not None and prev_alpha.shape == alpha.shape:
        alpha = ALPHA_TEMPORAL_WEIGHT*prev_alpha + (1-ALPHA_TEMPORAL_WEIGHT)*alpha
    prev_alpha = alpha.copy()
    alpha  = sharpen_alpha(alpha, SIGMOID_TEMP)
    alpha  = refine_alpha(alpha, MASK_CLOSE_PX, MASK_ERODE_PX, MASK_FEATHER_PX)
    f32    = rgb.astype(np.float32)/255.0
    f32    = white_despill(f32, alpha)
    f32    = decontaminate(f32, alpha)
    f32, alpha = remove_halo_dynamic(f32, alpha)
    fg_bgr = grade(cv2.cvtColor((f32*255).astype(np.uint8), cv2.COLOR_RGB2BGR))
    canvas = bg.copy()
    a3     = alpha[...,None]
    region = canvas[oy:oy+sh2, ox:ox+sw].astype(np.float32)
    canvas[oy:oy+sh2, ox:ox+sw] = np.clip(fg_bgr.astype(np.float32)*a3 + region*(1-a3), 0, 255).astype(np.uint8)
    canvas = add_rim_light(canvas, alpha, oy, ox, sh2, sw)
    canvas = bottom_fade(canvas)
    writer.write(canvas)

cap.release()
writer.release()
print('Combinando audio...')
subprocess.run(['ffmpeg','-y','-i',temp,'-i',INPUT_VIDEO,
    '-c:v','libx264','-preset','fast','-crf','18','-pix_fmt','yuv420p',
    '-c:a','aac','-b:a','192k','-map','0:v:0','-map','1:a:0?','-shortest',
    OUTPUT_VIDEO], check=True)
Path(temp).unlink(missing_ok=True)
elapsed = int(time.time()-t0)
size    = Path(OUTPUT_VIDEO).stat().st_size/1024/1024
print(f'OK -> {OUTPUT_VIDEO}  |  {elapsed}s  |  {size:.1f} MB')

### Ajuste fino si quedan halos

- `HEADREST_BRIGHTNESS`: bajar de 130 → 115 si quedan líneas blancas en hombros
- `HEADREST_Y_START / Y_END`: ajustar si los hombros están en otra posición del frame
- `FADE_START_Y`: bajar de 0.60 → 0.55 si la mesa sigue visible

In [ ]:
# CELDA 5 — Descargar resultado
# En Kaggle: el archivo queda en /kaggle/working/ — descárgalo desde la pestaña Output
# En Colab: se descarga automáticamente al navegador
import os
output_path = OUTPUT_VIDEO if 'OUTPUT_VIDEO' in dir() else 'video_dankoe_16x9.mp4'
if os.path.exists(output_path):
    size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f'Archivo listo: {output_path}  ({size_mb:.1f} MB)')
    print('Kaggle: descarga desde la pestaña Output (ícono carpeta → derecha)')
    try:
        from google.colab import files
        files.download(output_path)
        print('Colab: descarga iniciada')
    except ImportError:
        pass
else:
    print(f'No se encontró {output_path} — verifica que Celda 4 terminó sin errores')